In [1]:
import time
# autoreload
%load_ext autoreload
%autoreload 2

#from scipy import signal
#from scipy import interpolate
#from scipy import ndimage
import numpy as np
#import pycatch22 
#from sktime.transformations.panel import catch22
#import tsfresh
from tqdm import tqdm
import sys, os
import pandas as pd 
import dotenv
import random
load_dotenv = dotenv.load_dotenv('../.env')

# load local library
from timex import clustering
from timex import preprocessing

import gc
import seaborn as sns
import matplotlib.pyplot as plt

from collections import defaultdict

import datetime
from time import sleep

from aeon import datasets

/home/bramiozo/.cache/pypoetry/virtualenvs/timex-dV5LCb6l-py3.12/lib/python3.12/site-packages/nolds/datasets.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/bramiozo/.cache/pypoetry/virtualenvs/timex-dV5LCb6l-py3.12/lib/python3.12/site-packages/tslearn/metrics/dtw_variants.py:1749: SyntaxWarning: invalid escape sequence '\d'
  """Compute the mask (region constraint).


In [17]:
dataset = datasets.load_acsf1()

ts_data = pd.DataFrame(dataset[0].squeeze()).T
ts_label = dataset[1]

In [20]:
ts_data

,0,1,2,3,4,5,6,7,8,9,...,191,192,193,194,195,196,197,198,199,sample
0,-0.584754,-0.591434,-0.577945,-0.588925,-0.596633,-0.590045,-0.586470,-0.593479,-0.581512,-0.591501,...,-0.654323,-1.204223,-0.953089,-1.108539,-0.865824,-0.631937,-0.997077,-0.891590,-0.845868,0
1,-0.584754,-0.511104,-0.577945,-0.538088,-0.532188,-0.527073,-0.586470,-0.593479,-0.581512,-0.591501,...,-0.366839,0.566926,0.112185,0.272705,0.116534,-0.631937,0.108756,-0.752940,-0.650711,1
2,1.730991,1.726820,1.730793,1.735718,1.718067,1.753314,1.726075,1.744311,1.731894,1.716297,...,1.733656,1.509899,1.587598,1.496640,1.665577,1.612754,1.585963,1.424231,1.561223,2
3,-0.584754,-0.580422,-0.577945,-0.588716,-0.592117,-0.585371,-0.586470,-0.593479,-0.581512,-0.591501,...,-0.579071,-0.623772,-0.738562,-0.666800,-0.664466,-0.631937,-0.695920,-0.752324,-0.650859,3
4,-0.584754,-0.591434,-0.578946,-0.589962,-0.596633,-0.590045,-0.586470,-0.590336,-0.581512,-0.584326,...,-0.655383,-1.205211,-0.952067,-1.106615,-0.668138,-0.631937,-0.996043,-0.891590,-0.845868,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,-0.584734,-0.580731,-0.577751,-0.588876,-0.592403,-0.585847,-0.586226,-0.593124,-0.581512,-0.591104,...,-0.579723,-0.629160,-0.741305,-0.666680,-0.664985,-0.628213,-0.695931,-0.747132,-0.646646,1455
1456,-0.583729,-0.580731,-0.580956,-0.586852,-0.591524,-0.585847,-0.588574,-0.599776,-0.581512,-0.596624,...,-0.659623,-0.742400,-0.891688,-1.120098,-0.842507,-1.017398,-0.996043,-1.069707,-0.989189,1456
1457,-0.578603,-0.580731,-0.549798,-0.576483,-0.575158,-0.585847,-0.557007,-0.549405,-0.581512,-0.539187,...,-0.579009,-0.628675,-0.559115,0.294851,0.008064,0.088009,0.106678,0.481769,0.434901,1457
1458,1.732726,1.727396,1.734727,1.743664,1.743258,1.764487,1.755364,1.715375,1.712689,1.742528,...,1.718458,1.492112,1.601638,1.486969,1.672843,1.573765,1.589317,1.431407,1.553132,1458


In [ ]:
ts_data['sample'] = ts_data.index 
ts_data_unp = ts_data.melt(id_vars='sample', value_vars=list(range(ts_data.shape[1]-1)))\
                     .sort_values(by=['sample', 'variable'])
ts_data_unp = ts_data_unp.rename(columns = {'variable': 'Time'})

In [ ]:
# create heterogeinity
for s in ts_data_unp.sample.unique():
    max_t = ts_data_unp.loc[ts_data_unp.sample==s, 'Time'].max()
    # randomly subtract from max_t 
    new_max_t = max_t - np.random.randint(low=0, high=500)
    

,sample,Time,value
0,0,0,-0.584754
1460,0,1,-0.591434
2920,0,2,-0.577945
4380,0,3,-0.588925
5840,0,4,-0.596633
...,...,...,...
286159,1459,195,-0.664985
287619,1459,196,-0.628213
289079,1459,197,-0.695931
290539,1459,198,-0.747132


In [ ]:
MIN_TIME = 365 # days
MAX_TIME = 365*10 # days
MIN_MEAS_COUNT = 3 # measurements
INTERP_RES = 90
SMOOTHING_WINDOW = 4 # in days: 4 * INTERP_RES = 360
SMOOTHING_TYPE = 'gaussian_kernel'  # 'gaussian_kernel' or 'rolling_mean'
META_KEYS = ['sample', 'Time']
RAW_VAL_COL = 'value'

In [ ]:
ts_data_df = ts_data[['ID', 'Time_days', 'eGFRcr_CKDEpi2009']].dropna(subset=['eGFRcr_CKDEpi2009'])

In [ ]:
ts_clusterer = clustering.CrossSectionalClustering(smoothing=True, 
                                                   smoothing_type=SMOOTHING_TYPE,
                                                   smoothing_window_size=SMOOTHING_WINDOW,
                                                   n_skip=3,
                                                   interpolation=True, 
                                                   interpolation_resolution=INTERP_RES,
                                                   interpolation_keep_init=True,
                                                   min_measurements_per_id=MIN_MEAS_COUNT, 
                                                   min_time=MIN_TIME,
                                                   max_time=MAX_TIME,
                                                   n_clusters=3, 
                                                   id_column='ID', 
                                                   time_column='Time_days',
                                                   feature_columns=[RAW_VAL_COL],
                                                   imputation_method='knn',
                                                   cross_standardisation=True,
                                                   normalise_timeseries= "group",
                                                   normalisation_method="standard",
                                                   add_ts_meta=False,
                                                   verbose=True)

In [ ]:
ts_clusterer.fit(ts_data_df)

In [42]:
ts_temp = ts_clusterer.ts_filtered
ts_temp.loc[ts_temp.ID==3565782].head()

,ID,Time_days,eGFRcr_CKDEpi2009
1342,3565782,0.000000,108.503346
1347,3565782,4.102778,90.935916
1354,3565782,10.097917,88.845981
1360,3565782,15.073611,80.437337
1368,3565782,22.159028,84.919419


In [43]:
ts_temp = ts_clusterer.ts_interpolated
ts_temp.loc[ts_temp.ID==3565782].head()

,ID,Time_days,eGFRcr_CKDEpi2009
0,3565782,0,108.503346
1,3565782,90,109.304904
2,3565782,180,138.094540
3,3565782,270,133.880216
4,3565782,360,123.972524


In [44]:
ts_temp = ts_clusterer.ts_smoothed
ts_temp.loc[ts_temp.ID==3565782].head()


,ID,Time_days,eGFRcr_CKDEpi2009
0,3565782,0,108.503346
1,3565782,90,109.304904
2,3565782,180,138.094540
3,3565782,270,NaN
4,3565782,360,NaN


In [ ]:
ts_temp = ts_clusterer.ts_normalized
ts_temp.loc[ts_temp.ID==3565782].head()
